# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset on NC-21OHD-associated infertility using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the FAIR² Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

# Print dataset name and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

All entities (record sets, fields, columns) are referenced by their `@id`. Below we enumerate the available record sets and their structure.

In [ ]:
# Gather and inspect record sets and their fields
record_sets = []
record_set_fields = {}

for rs in dataset.metadata.recordSet:
    record_sets.append(rs['@id'])
    print(f"RecordSet @id: {rs['@id']} - {rs.get('name', 'No name')}")
    fields = []
    for field in rs.get('field', []):
        fields.append(field['@id'])
        print(f"  Field @id: {field['@id']} - {field.get('name', 'No name')}")
    record_set_fields[rs['@id']] = fields

# Print a sample record from each record set
for rs_id in record_sets:
    print(f"\nSample records from RecordSet @id: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    for i, record in enumerate(records[:1]):  # Show only first record as sample
        print(record)

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from all record sets
dataframes = {}

for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"\nFields in RecordSet {rs_id}: {df.columns.tolist()}")
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Below, we perform EDA on Table 2 (hormone levels, adrenal CT, gynecological ultrasound findings).

In [ ]:
# Example: Analyze hormone levels from Table 2
# Substitute actual @id values from the overview
# Let's assume Table 2 RecordSet @id is 'http://senscience.ai/table2' and column @id 'http://senscience.ai/hormone_lvl_X' is for '17-OHP'

# You may need to adjust the RecordSet and field IDs based on your data overview step
table2_rs_id = record_sets[1]  # Example: Table 2, check in your overview which is correct

# Example field/column @id for a hormone (from metadata or overview, update if needed)
hormone_field_id = record_set_fields[table2_rs_id][0]  # Use the first field as example

# Check types and filter numeric columns
df_table2 = dataframes[table2_rs_id]

if df_table2[hormone_field_id].dtype in [float, int]:
    threshold = 10  # Example threshold
    filtered_df = df_table2[df_table2[hormone_field_id] > threshold]
    print(f"Filtered records with {hormone_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the hormone field
    col_norm = f"{hormone_field_id}_normalized"
    filtered_df[col_norm] = (filtered_df[hormone_field_id] - filtered_df[hormone_field_id].mean()) / filtered_df[hormone_field_id].std()
    print(f"Normalized {hormone_field_id} for filtered records:")
    print(filtered_df[[hormone_field_id, col_norm]].head())

    # Group by another field/column
    group_field = record_set_fields[table2_rs_id][-1]  # Use last field as example (e.g. adrenal ultrasound finding)
    if group_field in df_table2.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())
else:
    print(f"Field {hormone_field_id} is not numeric. Available columns: {df_table2.columns.tolist()}")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. For example, plot hormone levels distribution from Table 2.

In [ ]:
# Visualization example: Histogram of hormone levels
if df_table2[hormone_field_id].dtype in [float, int]:
    plt.figure(figsize=(8,4))
    df_table2[hormone_field_id].hist(bins=10)
    plt.title(f"Distribution of {hormone_field_id} values")
    plt.xlabel(hormone_field_id)
    plt.ylabel("Count")
    plt.show()
else:
    print(f"Field {hormone_field_id} is not numeric. Unable to plot histogram.")

## 6. Conclusion
This notebook demonstrated loading, overviewing, extracting, and analyzing the FAIR² dataset using the `mlcroissant` library, referencing all entities by their `@id`. We explored sample record sets and applied EDA to hormone levels data, including normalization and grouping.

Key takeaways:
- The dataset is highly structured via the Croissant schema, with each entity uniquely identified by `@id`.
- Clinical and genetic data for rare disease cases can be loaded, visualized, and processed directly from the FAIR² schema.
- The limited sample size and patient diversity (N=3) restrict generalizability, but the dataset is valuable for illustrative and exploratory bioinformatics research.